## ADF Test (Stationarity Test - Raw Price)

In [2]:
import pandas as pd
import numpy as np

In [42]:
# Read file
dataset_df = pd.read_csv("dataset/df.csv")

dataset_df

,period,idr_usd
0,2015-01-02,12536.00
1,2015-01-05,12652.00
2,2015-01-06,12721.00
3,2015-01-07,12796.00
4,2015-01-08,12795.00
...,...,...
2445,2024-12-23,16351.35
2446,2024-12-24,16239.80
2447,2024-12-27,16289.04
2448,2024-12-30,16332.25


In [8]:
from statsmodels.tsa.stattools import adfuller

result = adfuller(dataset_df['idr_usd'])

print('ADF Statistic:', result[0])
print('p-value:', result[1])

ADF Statistic: -2.2829075968252988
p-value: 0.17753988132582138


ADF p-value < 0.05 -> Reject the null hypothesis (H0). The time series is stationary.
ADF p-value >= 0.05 -> Fail to reject the null hypothesis (H0). The time series is non-stationary.

Since the p-value is above the 5% significance value -> H0 is not rejected, the raw price is non-stationary.
-> Need log-transformation.

## GARCH Preprocessing: Convert to log-returns

In [31]:
# To ensure stationarity, apply log-return transformation

df_returns = pd.DataFrame()
df_returns['period'] = pd.to_datetime(dataset_df['period'])
df_returns['idr_usd'] = np.log(dataset_df['idr_usd'] / dataset_df['idr_usd'].shift(1))

# Remove NaN pertama due to shift(1)
df_returns = df_returns.dropna()

df_returns.head()

,period,idr_usd
1,2015-01-05,0.009211
2,2015-01-06,0.005439
3,2015-01-07,0.005878
4,2015-01-08,-0.000078
5,2015-01-09,-0.007216


## ADF (Stationarity) & Diagnostic Tests (Ljung, ARCH-LM, and JB)

In [32]:
result = adfuller(df_returns['idr_usd'])

print('ADF Statistic:', result[0])
print('p-value:', result[1])
print('Critical Values:', result[4])

ADF Statistic: -14.605087114685706
p-value: 4.107003685416438e-27
Critical Values: {'1%': np.float64(-3.4330328659222165), '5%': np.float64(-2.8627252631029694), '10%': np.float64(-2.5674009636186508)}


Since the p-value is less than the 5% significance value -> H0 is rejected, the usd/idr return is stationary and ready to be used as an input feature for the GARCH model.

In [33]:
# Autocorrelation test (Ljung-Box)
print("Ljung-Box")
from statsmodels.stats.diagnostic import acorr_ljungbox

for lag in [5, 10, 20]:
    lb_usd = acorr_ljungbox(df_returns['idr_usd'], lags=[lag], return_df=True)
    print(f"Lag {lag}: p-value = {lb_usd['lb_pvalue'].values[0]}")

print("=====================================================================================================")

# ARCH-LM test
from arch import arch_model
from statsmodels.stats.diagnostic import het_arch, normal_ad
from scipy.stats import jarque_bera

# ARCH-LM Test (lag=10)
lm_stat, lm_pvalue, _, _ = het_arch(df_returns['idr_usd'], nlags=10)
print("ARCH-LM Stat:", lm_stat)
print("ARCH-LM p-value:", lm_pvalue)

print("=====================================================================================================")

# Jarque-Bera Test (normalitas)
jb_stat, jb_pvalue = jarque_bera(df_returns['idr_usd'])
print("Jarque-Bera Stat:", jb_stat)
print("Jarque-Bera p-value:", jb_pvalue)

Ljung-Box
Lag 5: p-value = 8.466078017647456e-11
Lag 10: p-value = 2.6548312917368243e-10
Lag 20: p-value = 1.0638488331656142e-08
ARCH-LM Stat: 565.8578540366998
ARCH-LM p-value: 3.6156636477063053e-115
Jarque-Bera Stat: 5640.338282963202
Jarque-Bera p-value: 0.0


The result of the ARCH-LM Test shows that the p-value is below the 5% significance level, indicating the presence of volatility clustering, where current volatility depends on past volatility. Additionally, this confirms that the variance of the series is not constant (heteroskedastic), supporting the use of the GARCH model to capture time-varying volatility.

The result of the normality test displays that the p-value is below the 1% significance level, suggesting that the return distribution exhibits heavy tails and extreme values. Therefore, a <b>Student-t distribution</b> is more appropriate than the standard Gaussian distribution for modeling the error terms in the GARCH framework.

In [34]:
#Download file (returns)
df_returns.to_csv('df_returns.csv')

## Data Splitting (GARCH Modelling)

The dataset is split using a **time-based approach** to preserve the chronological order of observations. The dataset is divided into two subsets:
- **Training set (2015–2022):** used to generate volatility for LSTM's additional input feature.
- **Test set (2023–2024):** used to produce out-of-sample volatility predictions that are later utilized in the trading strategy simulation.

Since only the GARCH(1,1) model will be implemented, no comparison and out-of-sample evaluation are conducted for this model.

In [33]:
# Read file
df_returns = pd.read_csv("dataset/df_returns.csv")

df_returns = df_returns.drop(df_returns.columns[0], axis=1)
df_returns

,period,idr_usd
0,2015-01-05,0.009211
1,2015-01-06,0.005439
2,2015-01-07,0.005878
3,2015-01-08,-0.000078
4,2015-01-09,-0.007216
...,...,...
2444,2024-12-23,-0.000430
2445,2024-12-24,-0.006845
2446,2024-12-27,0.003027
2447,2024-12-30,0.002649


In [37]:
# Data splitting
df_returns['period'] = pd.to_datetime(df_returns['period'])

train = df_returns[(df_returns['period'] >= "2015-01-01") & (df_returns['period'] <= "2022-12-31")]
test  = df_returns[(df_returns['period'] >= "2023-01-01") & (df_returns['period'] <= "2024-12-31")]

In [7]:
train.head()

,period,idr_usd
0,2015-01-05,0.009211
1,2015-01-06,0.005439
2,2015-01-07,0.005878
3,2015-01-08,-0.000078
4,2015-01-09,-0.007216


In [9]:
test.tail()

,period,idr_usd
2444,2024-12-23,-0.000430
2445,2024-12-24,-0.006845
2446,2024-12-27,0.003027
2447,2024-12-30,0.002649
2448,2024-12-31,-0.005491


In [12]:
#Download file (train & test for GARCH Modelling)
train.to_csv('dataset/GARCH/train_garch.csv')
test.to_csv('dataset/GARCH/test_garch.csv')

## Data Preparation & Splitting - LSTM

Data Integration

In [14]:
# Read macro data
inflation_df = pd.read_csv("dataset/inflation_df.csv")
interest_df = pd.read_csv("dataset/interest_rate_df.csv")

In [25]:
# Merge data
lstm_df = pd.merge(dataset_df, inflation_df, on='period', how='outer')
lstm_df = pd.merge(lstm_df, interest_df, on='period', how='outer')

lstm_df[['inflation_rate', 'interest_rate']] = lstm_df[['inflation_rate', 'interest_rate']].ffill()
lstm_df = lstm_df.dropna() # Delete first row because idr_usd starts from 2 Jan

lstm_df.head()

,period,idr_usd,inflation_rate,interest_rate
1,2015-01-02,12536.0,6.96,7.75
2,2015-01-05,12652.0,6.96,7.75
3,2015-01-06,12721.0,6.96,7.75
4,2015-01-07,12796.0,6.96,7.75
5,2015-01-08,12795.0,6.96,7.75


In [26]:
lstm_df.tail()

,period,idr_usd,inflation_rate,interest_rate
2500,2024-12-23,16351.35,1.57,6.0
2501,2024-12-24,16239.80,1.57,6.0
2502,2024-12-27,16289.04,1.57,6.0
2503,2024-12-30,16332.25,1.57,6.0
2504,2024-12-31,16242.81,1.57,6.0


In [28]:
lstm_df.to_csv('dataset/LSTM/lstm_df.csv', index=False)

## Data Splitting

The split is defined as follows:
- **Training set (2015–2021):** used to train the LSTM model and learn the underlying patterns in the data.
- **Validation set (2022–2023):** used to monitor model performance during training and assist in hyperparameter tuning.
- **Test set (2023–2024):** used to evaluate the final model performance on unseen data.

Unlike the GARCH setup, which only requires a training and test split for volatility estimation, the LSTM model includes a validation set because deep learning models typically require an additional dataset to prevent overfitting and optimize model parameters.

In [15]:
# Read LSTM df (sudah di integrate, ada macro dan volatility)
lstm_df_train_val = pd.read_csv("dataset/LSTM/lstm_df_train_val.csv")

lstm_df_train_val

,period,idr_usd,inflation_rate,interest_rate,realized_return,cond_vol
0,2015-01-05,12652.00,6.96,7.75,0.009211,0.005032
1,2015-01-06,12721.00,6.96,7.75,0.005439,0.005578
2,2015-01-07,12796.00,6.96,7.75,0.005878,0.005540
3,2015-01-08,12795.00,6.96,7.75,0.000078,0.005552
4,2015-01-09,12703.00,6.96,7.75,0.007216,0.005242
...,...,...,...,...,...,...
2444,2024-12-23,16351.35,1.57,6.00,0.000430,0.004207
2445,2024-12-24,16239.80,1.57,6.00,0.006845,0.004207
2446,2024-12-27,16289.04,1.57,6.00,0.003027,0.004207
2447,2024-12-30,16332.25,1.57,6.00,0.002649,0.004207


In [16]:
lstm_df_train_val.index = pd.to_datetime(lstm_df_train_val['period'])

# Train
lstm_df_train = lstm_df_train_val[(lstm_df_train_val.index >= "2015-01-01") & (lstm_df_train_val.index <= "2021-12-31")]
# Val
lstm_df_val = lstm_df_train_val[(lstm_df_train_val.index >= "2022-01-01") & (lstm_df_train_val.index <= "2022-12-31")]
# Test
lstm_df_test  = lstm_df_train_val[(lstm_df_train_val['period'] >= "2023-01-01") & (lstm_df_train_val['period'] <= "2024-12-31")]

In [17]:
#Download file (train & test for GARCH Modelling)
lstm_df_train.to_csv('dataset/LSTM/train_lstm.csv', index=False)
lstm_df_val.to_csv('dataset/LSTM/val_lstm.csv', index=False)
lstm_df_test.to_csv('dataset/LSTM/test_lstm.csv', index=False)